# CoWork Enterprise Demo — Continuous Improvement
## Notebook 07 — Continuous Improvement (Self-Improving Loop)

_Icons used throughout: 🛠️ Action  📌 Note  🔹 Info_

> ⚠️ _Generated from `build_notebooks.py` — edit the builder and re-run. Direct edits to this notebook are overwritten._

---

### What This Notebook Does:

1. 🛠️ **Observes** real production usage to find failure patterns
2. 🛠️ **Mines** failing questions into the curated evaluation dataset
3. 🛠️ Runs an **Agent GPA baseline**, then **optimizes instructions only** (not tools)
4. 🛠️ **Re-evaluates** to prove the change helped with no regressions, then promotes

---

### Why a Self-Improving Agent Is Human-in-the-Loop:

🔹 Go-live is the start, not the end. A **self-improving agent** is a governed closed loop — **not** autonomous self-modification. You mine real usage for failures, score them with Agent GPA, improve the agent's **instructions** (the #1 lever — not more tools), and re-evaluate to prove it helped.

```
  observe traces  ->  mine failures  ->  curate eval set  ->  Agent GPA baseline
        ^                                                          |
        |                                                          v
   re-evaluate  <-  optimize INSTRUCTIONS only  <-  analyze failure patterns
```

📌 **Fast path (recommended): CoCo.** The Cortex Code CLI drives this whole loop with the `cortex-agent` skill — `dataset-curation` to mine traces, `evaluate-cortex-agent` to score, and `optimize-cortex-agent` to draft + apply improved instructions. This notebook shows the SQL-native equivalent so you can see and schedule each step.

---

### Estimated Time: **25–30 minutes**

### Prerequisites:
- Notebooks 00–05 complete (agent + registered eval dataset `DEMO_AGENT_EVALSET` + config stage)

In [ ]:
%%sql -r set_context
USE ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
USE DATABASE COWORK_ENTERPRISE_DEMO;
USE SCHEMA AGENTS;
USE WAREHOUSE COWORK_ENTERPRISE_DEMO_WH;


## Step 1: Observe Production Usage
Start from real interactions. Usage/latency/volume come from Account Usage; full step-by-step **traces** (planning, tool calls, errors) are in **AI & ML -> Agents -> DEMO_AGENT -> Monitoring** (AI Observability). Use those traces to spot wrong tool selection, redundant calls, and incomplete answers.

In [ ]:
%%sql -r observe_usage
SELECT START_TIME, USER_NAME, AGENT_NAME, TOKENS, TOKEN_CREDITS
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
WHERE AGENT_NAME = 'DEMO_AGENT'
ORDER BY START_TIME DESC
LIMIT 25;


## Step 2: Mine Failures into the Eval Set
In CoCo: *"Use the cortex agent dataset-curation skill to pull production traces for COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT and add the failing queries to COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_EVALSET with ground truth."* The SQL equivalent: append the previously-failing question (with ground truth) to the curated table, then re-register the dataset. Here we add a harder synthesis question that tends to expose redundant tool calls.

In [ ]:
%%sql -r add_failing_case
INSERT INTO COWORK_ENTERPRISE_DEMO.AGENTS.EVAL_QUESTIONS (INPUT_QUERY, GROUND_TRUTH)
  SELECT 'Which region has the highest AUM, and are there compliance concerns for clients there?',
    PARSE_JSON('{"ground_truth_output": "Identifies the top region by AUM from structured data AND summarizes any compliance concerns for that region from research notes, citing both sources. Should make one Analyst call and one Search call - not repeated calls.", "ground_truth_invocations": [{"tool_name": "nexus_analytics"}, {"tool_name": "nexus_research"}]}');


In [ ]:
%%sql -r reregister_eval_dataset
DROP DATASET IF EXISTS COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_EVALSET;
CALL SYSTEM$CREATE_EVALUATION_DATASET(
  'Cortex Agent',
  'COWORK_ENTERPRISE_DEMO.AGENTS.EVAL_QUESTIONS',
  'COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_EVALSET',
  OBJECT_CONSTRUCT('query_text', 'INPUT_QUERY', 'expected_tools', 'GROUND_TRUTH')
);


## Step 3: Baseline Evaluation (Agent GPA)
Run the scored evaluation from Notebook 05 against the updated dataset to establish a **baseline** for the current LIVE spec. (Uses the config staged in Notebook 05.)

```sql
CALL EXECUTE_AI_EVALUATION('START', OBJECT_CONSTRUCT('run_name', 'baseline'),
  '@COWORK_ENTERPRISE_DEMO.AGENTS.EVAL_CONFIG/agent_eval.yaml');
-- Poll STATUS until COMPLETED, then read scores + find the lowest-scoring queries:
SELECT INPUT, METRIC_NAME, EVAL_AGG_SCORE
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
  'COWORK_ENTERPRISE_DEMO', 'AGENTS', 'DEMO_AGENT', 'CORTEX AGENT', 'baseline'))
ORDER BY EVAL_AGG_SCORE ASC;
```

## Step 4: Optimize — Instructions Only
The lever is **instructions, not tools**. In CoCo: *"Based on the failure analysis, generate improved orchestration and response instructions... only update the instructions... apply the changes."* The SQL equivalent replaces the LIVE version's spec - here we add explicit efficiency and routing rules to `instructions.orchestration` and keep every tool identical.

> `MODIFY LIVE VERSION SET SPECIFICATION` is whole-spec replacement (omitted fields are removed), so we submit the full spec with only the instructions changed. Version the change with a COMMENT.

In [ ]:
%%sql -r optimize_instructions
ALTER AGENT COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT MODIFY LIVE VERSION SET SPECIFICATION =
$$
models:
  orchestration: auto

orchestration:
  budget:
    seconds: 45
    tokens: 16000

instructions:
  response: |
    You are the Enterprise Demo Analyst. You help portfolio managers, relationship managers,
    and compliance officers understand client portfolios, trading activity, and market research.

    Guidelines:
    - Be concise and data-driven. Lead with numbers when available.
    - When showing financial data, format large numbers (e.g., $2.5B not $2500000000).
    - If a question spans both structured data (portfolios, trades) and unstructured data
      (research notes), use BOTH tools to provide a complete answer.
    - Always cite which data source your answer came from.
    - For compliance questions, flag any potential issues clearly.
  orchestration: |
    - For questions about client AUM, portfolio values, positions, or trades: use the Analyst tool.
    - For questions about market outlook, research opinions, or compliance reports: use the Search tool.
    - For questions that need both, use both tools.
    - Prefer a single Analyst call for a purely numeric question; only add the Search tool
      when the question is qualitative, compliance-related, or explicitly asks "why".
    - Do not call the same tool twice for one question unless the user asks a follow-up.
  sample_questions:
    - question: "Who are our top 5 clients by AUM?"
    - question: "What is our total technology sector exposure?"
    - question: "Are there any compliance concerns I should know about?"
    - question: "Show me recent trades over $1M and the rationale behind them."

tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "nexus_analytics"
      description: "Query structured financial data including client accounts, portfolio positions, trade history, and business metrics. Use this for any question about numbers, rankings, aggregations, or trends."
  - tool_spec:
      type: "cortex_search"
      name: "nexus_research"
      description: "Search analyst research notes, market commentary, risk assessments, and compliance reports. Use this for qualitative insights, opinions, recommendations, and compliance flags."
  - tool_spec:
      type: "data_to_chart"
      name: "data_to_chart"
      description: "Generate visualizations from query results when the user asks for charts or visual breakdowns."

tool_resources:
  nexus_analytics:
    semantic_view: "COWORK_ENTERPRISE_DEMO.SEMANTIC.DEMO_SV"
    execution_environment:
      type: warehouse
      warehouse: "COWORK_ENTERPRISE_DEMO_WH"
  nexus_research:
    name: "COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_SEARCH"
    max_results: "5"
$$;
ALTER AGENT COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT SET COMMENT = 'v2 - instruction-only optimization (efficiency + routing rules)';


In [ ]:
%%sql -r verify_spec
DESCRIBE AGENT COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT;


## Step 5: Re-Evaluate and Compare
Re-run the same dataset and compare to baseline. Ship the change only if the target metric improves **and** there are no regressions on previously-passing queries.

```sql
CALL EXECUTE_AI_EVALUATION('START', OBJECT_CONSTRUCT('run_name', 'after_optimize'),
  '@COWORK_ENTERPRISE_DEMO.AGENTS.EVAL_CONFIG/agent_eval.yaml');
-- Side-by-side once both runs are COMPLETED:
WITH b AS (SELECT METRIC_NAME, AVG(EVAL_AGG_SCORE) s FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
    'COWORK_ENTERPRISE_DEMO','AGENTS','DEMO_AGENT','CORTEX AGENT','baseline')) GROUP BY 1),
  a AS (SELECT METRIC_NAME, AVG(EVAL_AGG_SCORE) s FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
    'COWORK_ENTERPRISE_DEMO','AGENTS','DEMO_AGENT','CORTEX AGENT','after_optimize')) GROUP BY 1)
SELECT b.METRIC_NAME, b.s AS BASELINE, a.s AS AFTER_OPT, a.s - b.s AS DELTA
FROM b JOIN a USING (METRIC_NAME) ORDER BY DELTA;
```

**If it improved:** promote the new spec to prod via Notebook 06 (the spec is the source of truth). **If not:** revert (`ALTER AGENT ... MODIFY LIVE VERSION SET SPECIFICATION = $$ <prior spec> $$`) and try a different instruction change. Schedule this loop (e.g. a monthly `TASK`) to keep the agent improving on real usage.

## What This Is (and Isn't)
- **Is:** a governed, auditable loop - every change is a versioned spec edit, gated by evaluation, promoted through dev->prod.
- **Isn't:** the agent editing itself. A human reviews the proposed instruction change and the before/after scores before anything ships.
- **Primary lever:** instruction quality and data/semantic-model quality - not adding tools.

## ✅ Notebook 07 Complete

### What You Just Built:
- A failing case mined into the eval dataset and re-registered
- An instruction-only optimization applied to the LIVE spec (tools unchanged), versioned by COMMENT
- A baseline-vs-after comparison pattern to prove improvement without regressions

---

### Key Points:
- Instructions - not more tools - are the primary improvement lever.
- Every change is a versioned, evaluated spec edit; a human approves before it ships.
- Schedule the loop (e.g. a monthly TASK) so the agent keeps improving on real usage.

---

### Next:

That completes the build lifecycle. Promote a validated improvement to prod via **Notebook 06**, and when you're done, run **Notebook 08** to clean up.